# Notebook for creating silhouette plots
Needs to be run after `..._clustering.ipynb`

In [ ]:
# Update path and working directory with project root
import os
import sys
from pathlib import Path

project_path = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file())
os.chdir(project_path)
sys.path.insert(0, str(project_path))

In [ ]:
# Imports
from pathlib import Path
from typing import cast

import pandas as pd
import plotnine as p9

from utils.plotting_utils import make_plot_silhouette
from utils.saving_utils import DatasetBundle, save_plot

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True

In [ ]:
# Load data
data_discov_full = DatasetBundle.load(Path("data/preprocessed/data_discov_full"))
data_discov_full.meta = cast(pd.DataFrame, data_discov_full.meta)
data_discov_full.na = cast(pd.DataFrame, data_discov_full.na)
data_valid_full = DatasetBundle.load(Path("data/preprocessed/data_valid_full"))
data_valid_full.meta = cast(pd.DataFrame, data_valid_full.meta)
data_valid_full.na = cast(pd.DataFrame, data_valid_full.na)

data_discov_mfa = pd.read_parquet(Path("data/dim_red/data_discov_mfa.parquet"))
data_valid_mfa = pd.read_parquet(Path("data/dim_red/data_valid_mfa.parquet"))

proba_and_labels_discov = pd.read_parquet(Path("outputs/gmm_clustering/cluster_pred_discov.parquet"))
proba_and_labels_valid = pd.read_parquet(Path("outputs/gmm_clustering/cluster_pred_valid.parquet"))

# Set outputs folder
output_folder = Path("outputs/gmm_clustering")
output_folder.mkdir(exist_ok=True)

# Set predicted probability and labels
proba_discov = proba_and_labels_discov.drop("label", axis="columns")
proba_valid = proba_and_labels_valid.drop("label", axis="columns")
labels_discov = proba_and_labels_discov["label"]
labels_valid = proba_and_labels_valid["label"]

# Set sample weights
sample_weight_discov = data_discov_full.data["wichtn"].to_numpy()
sample_weight_valid = data_valid_full.data["wichtn"].to_numpy()

In [ ]:
# Make silhouette plots
silhouette_plot_discov = make_plot_silhouette(
    data=data_discov_mfa, cluster_labels=labels_discov, sample_weight=sample_weight_discov
)
silhouette_plot_valid = make_plot_silhouette(
    data=data_valid_mfa, cluster_labels=labels_valid, sample_weight=sample_weight_valid
)

# Combine silhouette plots
silhouette_plot_all = (
    (silhouette_plot_discov + p9.labs(title="Discovery") + p9.theme(legend_position="none"))
    + (silhouette_plot_valid + p9.labs(title="Validation"))
) & p9.theme(figure_size=(10, 4))
silhouette_plot_all.show()

# Save plots
# save_plot(silhouette_plot_discov, save_path=output_folder / "silhouette_plot_discov.png", dpi=150)
# save_plot(silhouette_plot_valid, save_path=output_folder / "silhouette_plot_valid.png", dpi=150)

save_plot(silhouette_plot_all, save_path=output_folder / "silhouette_plot_all.png", dpi=150)